# **Governance** **Officer**



In [ ]:
!pip install mongomock pymongo
import json
import mongomock
import hashlib
import pandas as pd

In [ ]:
client = mongomock.MongoClient()
db = client.credit_audit
collection = db.applications

# Load JSON file
with open('raw_credit_applications.json') as f:
    file_data = json.load(f)

# Ensure uniqueness of documents based on '_id' before insertion.
# This will keep the first occurrence of a document if its '_id' is duplicated in the source file.
seen_ids = set()
unique_documents = []
for doc in file_data:
    if '_id' in doc:
        if doc['_id'] not in seen_ids:
            unique_documents.append(doc)
            seen_ids.add(doc['_id'])
        else:
            # Optionally, you can print a message for skipped duplicates
            print(f"Skipping duplicate document with _id: {doc['_id']} from the source file.")
    else:
        # If a document doesn't have an _id, MongoDB will generate one, so it won't cause this specific duplicate key error.
        unique_documents.append(doc)

# Clear existing data to prevent duplicate key errors from previous runs
collection.delete_many({})

# Insert on NoSQL
try:
    if unique_documents:
        collection.insert_many(unique_documents)
        print(f"Dataset loaded: {collection.count_documents({})} records ready for audit.")
    else:
        print("No unique documents found to insert.")
except Exception as e:
    print(f"An error occurred during insertion: {e}")

Skipping duplicate document with _id: app_042 from the source file.
Skipping duplicate document with _id: app_001 from the source file.
Dataset loaded: 500 records ready for audit.


## Identifying PII

As the first fundamental step of a Data Governance strategy, we must conduct a systematic Data Discovery process. It is impossible to protect or govern data whose existence, location, and format are unknown. This cell implements a NoSQL search engine (using mongomock to simulate a document-oriented environment) to audit the raw_credit_applications.json dataset.

The technical objective is to map every field containing Personally Identifiable Information (PII) and verify its current protection status. According to the GDPR and the EU AI Act, precise identification of these fields is a legal prerequisite to ensure the credit decision system operates within regulatory boundaries.

In [ ]:
import json
import mongomock

# 1. Initialize the Mock NoSQL Database
client = mongomock.MongoClient()
db = client.governance_audit
collection = db.credit_data

# Load the dataset
with open('raw_credit_applications.json', 'r') as file:
    file_data = json.load(file)

# Ensure uniqueness of documents based on '_id' before insertion.
# This will keep the first occurrence of a document if its '_id' is duplicated in the source file.
seen_ids = set()
unique_documents = []
for doc in file_data:
    if '_id' in doc:
        if doc['_id'] not in seen_ids:
            unique_documents.append(doc)
            seen_ids.add(doc['_id'])
        else:
            print(f"Skipping duplicate document with _id: {doc['_id']} from the source file.")
    else:
        unique_documents.append(doc)

# Clear existing data to prevent duplicate key errors from previous runs
collection.delete_many({})

# Insert on NoSQL
try:
    if unique_documents:
        collection.insert_many(unique_documents)
        print(f"Dataset loaded: {collection.count_documents({})} records ready for audit.")
    else:
        print("No unique documents found to insert.")
except Exception as e:
    print(f"An error occurred during insertion: {e}")

# 2. Define the PII Dictionary based on GDPR categories
pii_inventory = {
    "Direct Identifiers": [
        "applicant_info.full_name",
        "applicant_info.email",
        "applicant_info.ssn"
    ],
    "Online Identifiers": [
        "applicant_info.ip_address"
    ],
    "Quasi-Identifiers / Demographic": [
        "applicant_info.date_of_birth",
        "applicant_info.zip_code",
        "applicant_info.gender"
    ]
}

print("___________________________________________________\n")
print("PII identification")
print("___________________________________________________\n")

total_records = collection.count_documents({})
print(f"Total documents scanned: {total_records}\n")

# 3. Execute NoSQL Queries to prove PII exists
for category, fields in pii_inventory.items():
    print(f"{category}")

    for field in fields:
        # NoSQL Query: Find documents where the field exists AND is not an empty string
        query = {field: {"$exists": True, "$nin": ["", None]}}
        exposure_count = collection.count_documents(query)

        if exposure_count > 0:
            # Fetch one example to prove the exposure in the audit report
            example_doc = collection.find_one(query, {"_id": 1, field: 1})

            # Navigate the nested JSON safely to extract the specific value
            keys = field.split('.')
            exposed_value = example_doc
            for k in keys:
                exposed_value = exposed_value.get(k, {})

            print(f"[!] FOUND: '{field}' is exposed in {exposure_count} records.")
            print(f"Evidence (Doc _id: {example_doc['_id']}): {exposed_value}")
        else:
            print(f"[OK] '{field}' is securely blank or missing.")
    print("\n")

Skipping duplicate document with _id: app_042 from the source file.
Skipping duplicate document with _id: app_001 from the source file.
Dataset loaded: 500 records ready for audit.
___________________________________________________

PII identification
___________________________________________________

Total documents scanned: 500

Direct Identifiers
[!] FOUND: 'applicant_info.full_name' is exposed in 500 records.
Evidence (Doc _id: app_200): Jerry Smith
[!] FOUND: 'applicant_info.email' is exposed in 493 records.
Evidence (Doc _id: app_200): jerry.smith17@hotmail.com
[!] FOUND: 'applicant_info.ssn' is exposed in 496 records.
Evidence (Doc _id: app_200): 596-64-4340


Online Identifiers
[!] FOUND: 'applicant_info.ip_address' is exposed in 496 records.
Evidence (Doc _id: app_200): 192.168.48.155


Quasi-Identifiers / Demographic
[!] FOUND: 'applicant_info.date_of_birth' is exposed in 496 records.
Evidence (Doc _id: app_200): 2001-03-09
[!] FOUND: 'applicant_info.zip_code' is exposed i

The execution of the identification script successfully mapped the data inventory of the raw_credit_applications.json dataset, revealing critical compliance flaws. As Data Governance Officer, the interpretation of these results in light of the GDPR and the EU AI Act demonstrates that the organization is in a state of legal and ethical vulnerability. Firstly, a high risk of direct identification was detected due to the exposure of names, emails, and especially the Social Security Number (SSN) in plain text. Storing the SSN without any hashing or encryption technique directly violates Article 32 of the GDPR on the security of processing and Article 87 concerning national identification numbers, representing an extreme risk of identity theft that requires the immediate implementation of Field-Level Encryption (FLE).

Additionally, the audit identified the collection of online and geographic identifiers, such as IP addresses and postal codes. Under the Data Minimization principle (Article 5), the retention of IP addresses without a clear justification for fraud prevention is excessive. The presence of postal codes in an automated decision-making system is particularly dangerous, as it can facilitate "redlining" or geographic profiling, where the algorithm discriminates against candidates based on their neighborhood of residence instead of individual financial merit.

In the context of the EU AI Act, this system is classified as High Risk (Annex III) because it deals with credit scoring. The presence of fields such as gender and date of birth requires rigorous bias monitoring to ensure that the algorithm does not discriminate against protected groups. Furthermore, the format inconsistencies found in the dates of birth violate the data quality standards required by Article 10 of the AI ​​Act for high-risk AI systems. In conclusion, the ease with which this data was retrieved via NoSQL proves the total absence of technical safeguards, such as pseudonymization. This audit therefore serves as evidence of non-compliance and as a basis for the urgent recommendation of Privacy by Design controls, technical documentation for certification, and the undertaking of a Data Protection Impact Assessment (DPIA).

##Privacy Transformation

Next we will target the email and SSN columns. We will replace the raw values with a secure hash. This proves that the system can still function (e.g., checking if the same person applied twice) without storing the actual PII.

In [ ]:
import hashlib

# 1. Function to create a pseudonym (Hash)
def pseudonymize(value):
    if not value:
        return None
    # Using SHA-256 to create a unique fingerprint of the data
    return hashlib.sha256(value.encode()).hexdigest()

# 2. Apply Pseudonymization to the NoSQL Collection
print("Starting Privacy Transformation\n")

# We iterate through the collection and update each record
for doc in collection.find():
    # Target fields: email and ssn
    raw_email = doc['applicant_info'].get('email')
    raw_ssn = doc['applicant_info'].get('ssn')

    pseudo_email = pseudonymize(raw_email)
    pseudo_ssn = pseudonymize(raw_ssn)

    # Update the document in the mock NoSQL DB
    collection.update_one(
        {"_id": doc["_id"]},
        {"$set": {
            "applicant_info.email": pseudo_email,
            "applicant_info.ssn": pseudo_ssn,
            "privacy_status": "pseudonymized"
        }}
    )


print(f"[EVIDENCE] Transformation for Record ID: {target_id}")
print(f"Algorithm used: SHA-256 (Cryptographic Hashing)")
print("_" * 80)
print(f"FIELD: Email\n")
print(f"  AFTER : {after_doc['applicant_info']['email']}")
print("_" * 80)
print(f"FIELD: SSN (Sensitive Government ID)\n")
print(f"  AFTER : {after_doc['applicant_info']['ssn']}")
print("_" * 80)

Starting Privacy Transformation

[EVIDENCE] Transformation for Record ID: app_200
Algorithm used: SHA-256 (Cryptographic Hashing)
________________________________________________________________________________
FIELD: Email

  AFTER : 89f32645fc458895d8fc51666b41e3693e48e118734d6219732f068b18200504
________________________________________________________________________________
FIELD: SSN (Sensitive Government ID)

  AFTER : f3d9f79a6486f2cb4d6a5e100f3bb11f78e68a52385c39f9d958e9f63a90bb94
________________________________________________________________________________


The output above confirms the successful implementation of cryptographic pseudonymization using the SHA-256 algorithm. By transforming sensitive fields (specifically the Email and SSN) into unique hexadecimal hashes, we have effectively de-coupled the data subject's identity from the processing activity.

This transformation directly fulfills the GDPR requirements of Article 32 (Security of Processing) and Article 25 (Data Protection by Design and by Default). Even in the event of unauthorized access to the NoSQL collection, the core PII remains protected and unreadable.

Unlike full anonymization, this pseudonymized state preserves "referential integrity." This means the credit scoring algorithm can still identify patterns or repeat applicants by comparing hashes, without ever needing to "know" the actual SSN.

This process demonstrates a transition from a high-risk data state to a protected governance framework, significantly reducing the organization's legal liability and protecting the data subjects' fundamental rights.

In [ ]:
print("Anonymization Audit: Data Minimization Process ")

# 1. Defining fields for complete anonymization
# IP addresses and alcohol expenses are considered excessive for this purpose.
fields_to_anonymize = ["applicant_info.ip_address"]


for doc in collection.find():
    # Remove IP Address
    collection.update_one(
        {"_id": doc["_id"]},
        {"$unset": {"applicant_info.ip_address": ""}}
    )

    # Filter the spending_behavior array to remove sensitive categories (Alcohol)
    collection.update_one(
        {"_id": doc["_id"]},
       {"$pull": {"spending_behavior": {"category": "Alcohol"}}}
    )

sample_after = collection.find_one({"_id": "app_200"})

print(f"\n[EVIDENCE] Anonymization for Record ID: app_200")
print("_" * 80)
print(f"IP Address field exists?  {'ip_address' in sample_after['applicant_info']}")
print(f"Sensitive 'Alcohol' category exists? ", any(item['category'] == 'Alcohol' for item in sample_after['spending_behavior']))
print("_" * 80)
print("\n[Note]: Irreversible anonymization complete. Excess data removed per GDPR Art. 5(1)(c).")

Anonymization Audit: Data Minimization Process 

[EVIDENCE] Anonymization for Record ID: app_200
________________________________________________________________________________
IP Address field exists?  False
Sensitive 'Alcohol' category exists?  False
________________________________________________________________________________

[Note]: Irreversible anonymization complete. Excess data removed per GDPR Art. 5(1)(c).


Anonymization permanently removes data that is not strictly necessary for the purpose of credit scoring. This aligns with Article 5.1(c) (Data Minimization). Since IP addresses and behavioral data like "Alcohol consumption" are not essential for financial risk assessment, they were irreversibly removed to protect the subject's private life.

##Proposed Governance Controls & Policy Recommendations

Based on our findings, the following concrete actions are proposed to move the Credit Scoring system from a high-risk state to a compliant, ethical framework:


1) Establishment of Audit Trails:

A dedicated logging collection must be implemented in the NoSQL database. Every automated decision must record the model version used, the timestamp, and the specific input features (pseudonymized). This ensures "Traceability," a core requirement of the EU AI Act.

---
2) Human Oversight Protocol (**Art. 14 AI Act**):




For loan rejections triggered by high-risk scores, a "Human-in-the-loop" flag must be added. This requires a human credit officer to review and sign off on the rejection, preventing 100% automated bias.

---
3) Active Consent Mechanism (**Art. 7 GDPR**):




The JSON schema must be updated to include a consent_metadata object. This should store the date of consent, the specific purposes accepted (e.g., "credit assessment" vs. "marketing"), and the version of the privacy policy agreed to at the time of application.

---

4) Automated Data Retention Policy (**Art. 5.1.e GDPR**):




A retention_expiry field must be added to every record. We recommend a 5-year retention period for rejected applications and 10 years for active loans. A "Time-to-Live" (TTL) index should be configured in the database to automatically delete records once they expire, ensuring permanent storage limitation.